# VAE solver — training, posterior ranking, and latent activations

A walkthrough of the **conditional VAE** that tries to place tetrominoes on the
puzzle grid. Use this notebook to present the method, run a tiny training loop,
and **probe the latent network**: CNN feature maps, $z$ posteriors, and how
ranking changes when you swap pieces or board scenarios.

**Canonical model:** VAE v2 (posterior ranking). Shipped weights live in
`models/vae_clean/`. Full training is `uv run train-vae` (writes
`models/vae_v2_trained/` and does not overwrite the shipped model).

| | |
|---|---|
| **Idea** | Encode `(board, action) → z`, decode `(board, z, piece) → action` |
| **Play** | Rank every legal placement by reconstruction loss under $q(z \mid board, action)$ |
| **Headline result** | Reconstructs demos well, **clears ~0 lines** in play (covariate shift) |

Kernel: the package `.venv` (run `uv sync --group notebooks` first).


## 0. Setup

Import `nb_setup` **before** TensorFlow so CUDA is pinned to CPU. Matplotlib
and the engine/solver modules are then on `sys.path`.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

nb_dir = Path.cwd()
if not (nb_dir / "nb_setup.py").exists():
    nb_dir = nb_dir / "notebooks"
sys.path.insert(0, str(nb_dir))

import nb_setup as N
N.setup_notebook()

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from puzzlegrid.pieces import Piece, PieceType
from puzzlegrid import placement
from training_vae_v2 import (
    extract_board_and_actions,
    load_training_data,
    subsample_training_data,
    build_tetris_vae,
    train_model,
)

print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


## 1. What the VAE is learning

The VAE is a **behavioral clone**, not a value function.

1. A CNN compresses the 20×10 occupancy grid to `board_feat` (256-d).
2. An action encoder maps `(board_feat, piece, rotation, position) → (z_mean, z_log_var)`.
3. A decoder reconstructs the action from `(board_feat, z, piece)`.
4. Training minimizes reconstruction loss + $\beta$-KL (with $\beta$ annealed).

**v1 vs v2.** v1 sampled $z \sim \mathcal{N}(0,I)$ at play time while training
used the posterior — a train/inference mismatch. v2 ranks *legal* placements by
how well the posterior reconstructs them. That is the right inference procedure.
It still fails at play because imitation has no notion of “this board is dead.”

Read: `docs/VAE_writeup.md`, `docs/vae_explaination_and_improvement_plan.md`.


## 2. Peek at the teaching data


In [ ]:
game = N.load_game()
print("file keys:", sorted(game.keys())[:12])
print("size:", game.get("width"), "x", game.get("height"),
      "lines_cleared:", game.get("lines_cleared") or game.get("total_lines_cleared"))
places = [m for m in game["moves"] if m.get("move_type") == "place"]
print("place moves:", len(places), "first piece:", places[0].get("piece_type"))

board, piece = N.board_before_nth_place(game, n=12)
print("scenario piece at move 12:", piece)
N.show_board(board, title=f"Real game, before move 12 ({piece})")


In [ ]:
# Small extract so we can see label shapes without loading all 191 games.
sample_games = load_training_data(str(N.DATA_DIR), max_files=3)
data = extract_board_and_actions(sample_games)
print({k: np.shape(v) for k, v in data.items()})
print("piece histogram:", dict(zip(*np.unique(data["piece_idx"], return_counts=True))))
print("rotation histogram:", dict(zip(*np.unique(data["rotation_idx"], return_counts=True))))
print("line-clearing fraction:", float(np.mean(data["line_clears"] > 0)))


## 3. Architecture (v2)

Board CNN keeps spatial layout (strided convs + flatten — not global average
pool). Latent is 64-d. Decoder has separate heads for piece, rotation, and
normalized `(row, col)`.

```
board (20×10×1)
  Conv 32 → 64 → 96 (stride 2,1) → 128 (stride 2,2) → flatten → Dense 256
                                                     board_feat
action = concat[piece(7), rotation(4), position(2)]     (13)
encoder MLP: concat(board_feat, action) → z_mean, z_log_var  (64)
decoder: concat(board_feat, z, piece_embed) → piece / rotation / position
```


In [ ]:
# Fresh untrained model so we can print a summary without loading 46MB yet.
demo = build_tetris_vae(latent_dim=64, beta=0.05)
_ = demo.board_encoder(tf.zeros((1, 20, 10, 1)))
demo.board_encoder.summary()


## 4. Optional short training

Leave `RUN_SHORT_TRAINING = False` for a presentation that uses shipped weights
(seconds). Set it `True` to fit a **tiny** subset in-notebook so you can show
the training API. Full training: `uv run train-vae`.


In [ ]:
RUN_SHORT_TRAINING = False  # flip to True for a 1-epoch demo

if RUN_SHORT_TRAINING:
    games = load_training_data(str(N.DATA_DIR), max_files=4)
    data = subsample_training_data(extract_board_and_actions(games), max_samples=512)
    model = build_tetris_vae(latent_dim=64, beta=0.05, learning_rate=1e-3)
    history = train_model(
        model, data,
        epochs=1, batch_size=32,
        model_dir=str(N.REPO_ROOT / "models" / "vae_notebook_demo"),
        beta_anneal_epochs=1,
        early_stopping_patience=5,
    )
    print({k: v[-1] for k, v in history.history.items()})
else:
    print("Skipping in-notebook training. Loading shipped models/vae_clean/ next.")


## 5. Load the shipped VAE

The saved `TetrisVAEV2` class does not deserialize (encoder/decoder are not in
`get_config`). We rebuild the class and `load_weights` from
`tetris_vae_v2_best.keras` — same weights the replay client wants.


In [ ]:
vae = N.load_vae_v2()
print("latent_dim", vae.latent_dim, "beta", vae.beta)
print("board encoder", vae.board_encoder.name, "decoder", vae.decoder.name)

conv_names = ["conv2d", "conv2d_1", "conv2d_2", "conv2d_3"]
cnn_acts = N.conv_activation_model(vae.board_encoder, conv_names)


## 6. Experiment lab — pick a piece and a scenario

Edit `PIECE` and `SCENARIO` and re-run from here. The rest of the notebook
uses these. Try:

- `almost_line` + `I` (horizontal) — a free line clear the VAE often still misses
- `right_well` + `I` — the Tetris well
- `bumpy` + `T` — off-distribution surface
- `empty` + any piece — spawn-like board


In [ ]:
PIECE = "I"          # I O T S Z J L
SCENARIO = "almost_line"  # empty almost_line right_well left_well bumpy flat_stack overhang_pocket

board = N.SCENARIOS[SCENARIO]()
piece_type = PieceType[PIECE]
candidates = placement.get_valid_placements(board, piece_type)
print(f"{SCENARIO} / {PIECE}: {len(candidates)} legal reachable placements")
N.show_board(board, title=f"{SCENARIO} — current piece {PIECE}")


## 7. CNN activations (the spatial latent)

Each conv layer is a stack of feature maps. Watch how early layers light up
occupied cells and later layers smear that into wells / holes / row-fill
patterns. Change the scenario above and re-run to see the difference.


In [ ]:
board_in = N.encode_board(board)
maps = cnn_acts.predict(board_in, verbose=0)
for name, m in zip(conv_names, maps):
    N.show_feature_maps(m, title=f"board_encoder.{name}  shape={tuple(m.shape)}", max_channels=16)


## 8. Posterior $z$ for every legal placement

For each candidate action we encode $q(z \mid board, action)$ and decode.
v2 **ranks by reconstruction loss** (piece CE + rotation CE + 10 × position MSE).
Lower is “more like a demonstrated placement.”


In [ ]:
def vae_score_candidates(model, board, piece_name):
    piece = PieceType[piece_name]
    cands = placement.get_valid_placements(board, piece)
    if not cands:
        return [], None, None
    h, w = len(board), len(board[0])
    n = len(cands)
    feat = model.board_encoder(N.encode_board(board), training=False)
    feat_b = tf.repeat(feat, n, axis=0)
    piece_idx = np.full((n,), N.PIECE_NAMES.index(piece_name), np.int32)
    rot = np.array([p[2] for p in cands], np.int32)
    pos = np.array([[p[0] / h, p[1] / w] for p in cands], np.float32)
    action = model.build_action_vector(
        tf.constant(piece_idx), tf.constant(rot), tf.constant(pos)
    )
    z_mean, z_log_var, _ = model.encode(feat_b, action, training=False)
    piece_oh = tf.one_hot(piece_idx, 7, dtype=tf.float32)
    piece_logits, rot_logits, pos_pred = model.decode(feat_b, z_mean, piece_oh, training=False)
    piece_loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=piece_idx, logits=piece_logits
    ).numpy()
    rot_loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=rot, logits=rot_logits
    ).numpy()
    pos_loss = tf.reduce_mean(tf.square(pos_pred - pos), axis=-1).numpy()
    recon = (
        model.piece_loss_weight * piece_loss
        + model.rotation_loss_weight * rot_loss
        + model.position_loss_weight * pos_loss
    )
    lines = []
    for (r, c, rot_) in cands:
        _, lc = placement.simulate_placement(board, piece, r, c, rot_)
        lines.append(lc)
    rows = []
    for i, (r, c, rot_) in enumerate(cands):
        rows.append({
            "i": i, "row": r, "col": c, "rot": rot_,
            "recon": float(recon[i]),
            "piece_ce": float(piece_loss[i]),
            "rot_ce": float(rot_loss[i]),
            "pos_mse": float(pos_loss[i]),
            "lines": int(lines[i]),
            "z_norm": float(np.linalg.norm(z_mean[i])),
        })
    rows.sort(key=lambda d: d["recon"])
    return rows, z_mean.numpy(), z_log_var.numpy()

ranked, z_mean, z_log_var = vae_score_candidates(vae, board, PIECE)
print(f"{len(ranked)} candidates, best recon={ranked[0]['recon']:.4f}")
N.ranked_table(ranked, ["recon", "col", "row", "rot", "lines", "piece_ce", "rot_ce", "pos_mse", "z_norm"], n=15)


In [ ]:
best = ranked[0]
worst = ranked[-1]
N.show_boards(
    [
        {"board": board, "piece": (best["row"], best["col"], best["rot"]),
         "piece_type": PIECE, "title": f"VAE pick  recon={best['recon']:.3f}  lines={best['lines']}"},
        {"board": board, "piece": (worst["row"], worst["col"], worst["rot"]),
         "piece_type": PIECE, "title": f"worst recon={worst['recon']:.3f}  lines={worst['lines']}"},
    ],
    cols=2,
)

# Reconstruction loss as a column heatmap (min over rotations that land in that column).
heat = np.full((20, 10), np.nan)
for row in ranked:
    r, c = row["row"], row["col"]
    if np.isnan(heat[r, c]) or row["recon"] < heat[r, c]:
        heat[r, c] = row["recon"]
fig, ax = plt.subplots(figsize=(4.2, 7.4))
N.show_board(board, heatmap=heat, title="posterior recon loss (lower = more VAE-like)", ax=ax, show=False)
plt.tight_layout(); plt.show()


## 9. Look at $z$ itself

`z_mean` is 64-d. We plot (a) the chosen placement’s latent as an 8×8 grid,
(b) per-dimension variance across candidates, (c) a PCA of all candidate
posteriors colored by column — so you can see whether the latent organizes
*where* the piece goes.


In [ ]:
order = [r["i"] for r in ranked]
z_ord = z_mean[order]
z_best = z_ord[0]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].imshow(z_best.reshape(8, 8), cmap="coolwarm", origin="upper")
axes[0].set_title("z_mean of VAE pick (8×8)")
axes[0].set_xticks([]); axes[0].set_yticks([])

std = z_mean.std(axis=0)
axes[1].bar(np.arange(len(std)), std, color="#67e8f9")
axes[1].set_title("std(z) across candidates")
axes[1].set_xlabel("latent dim")

from numpy.linalg import svd
Xz = z_mean - z_mean.mean(axis=0, keepdims=True)
_, _, Vt = svd(Xz, full_matrices=False)
xy = Xz @ Vt[:2].T
cols = np.array([p[1] for p in candidates])
sc = axes[2].scatter(xy[:, 0], xy[:, 1], c=cols, cmap="viridis", s=28)
axes[2].set_title("PCA of z_mean (color = column)")
plt.colorbar(sc, ax=axes[2], fraction=0.046, label="col")
plt.tight_layout(); plt.show()

print("mean |z_mean|", float(np.mean(np.abs(z_mean))),
      "mean exp(0.5 log_var) ~ posterior std",
      float(np.mean(np.exp(0.5 * z_log_var))))


## 10. Latent-dimension sweep

Hold the VAE's chosen $z$ and walk **one coordinate** from -3 to +3. Decode
the resulting `(rotation, position)`. If a dimension is used, the decoded
column/rotation will move; if it is dead, nothing happens. Change `DIM`.


In [ ]:
DIM = int(np.argmax(std))  # most active dim by default; try 0..63
print("sweeping latent dim", DIM, "std", float(std[DIM]))

feat = vae.board_encoder(N.encode_board(board), training=False)
piece_oh = N.piece_onehot(PIECE)
base = z_best.copy()
vals = np.linspace(-3, 3, 9)
decoded = []
for v in vals:
    z = base.copy()
    z[DIM] = v
    z_in = z.reshape(1, -1).astype(np.float32)
    p_logits, r_logits, pos = vae.decode(feat, z_in, piece_oh, training=False)
    rot = int(np.argmax(r_logits.numpy()[0]))
    row = int(round(float(pos[0, 0]) * 20))
    col = int(round(float(pos[0, 1]) * 10))
    decoded.append((v, row, col, rot))

print("value  row  col  rot")
for v, r, c, rot in decoded:
    print(f"{v:6.2f}  {r:3d}  {c:3d}  {rot:3d}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot([d[0] for d in decoded], [d[2] for d in decoded], "o-", label="decoded col")
ax.plot([d[0] for d in decoded], [d[3] for d in decoded], "s-", label="decoded rot")
ax.set_xlabel(f"z[{DIM}]"); ax.legend(); ax.set_title("decoder response to a latent walk")
plt.tight_layout(); plt.show()


## 11. Solver step (in-process)

Place the VAE’s ranked choice, show the resulting board, and compare against
the candidate that actually clears the most lines (oracle among legal moves).
This is the v2 replay rule, without the HTTP server.


In [ ]:
pick = (best["row"], best["col"], best["rot"])
oracle = max(ranked, key=lambda d: (d["lines"], -d["recon"]))
oracle_p = (oracle["row"], oracle["col"], oracle["rot"])
post_vae, lc_vae = placement.simulate_placement(board, piece_type, *pick)
post_or, lc_or = placement.simulate_placement(board, piece_type, *oracle_p)

N.show_boards(
    [
        {"board": post_vae, "title": f"after VAE pick  lines+={lc_vae}"},
        {"board": post_or, "title": f"after line-clear oracle  lines+={lc_or}"},
    ],
    cols=2,
)
print("VAE chose", pick, "oracle", oracle_p)
print("Same move?" , pick == oracle_p)


## 12. Takeaways for a presenter

- **Training works.** Piece/rotation accuracy on held-out *demonstrations* is
  excellent. That is necessary and not sufficient.
- **Inference is posterior ranking**, not prior sampling. That was the v1 bug.
- **Activations are spatially meaningful** — the CNN sees occupancy — but the
  latent is organized around *imitating a placement*, not *valuing a board*.
- On `almost_line`, check whether the lowest recon loss is actually the clear.
  Often it is not: the teacher’s typical I-placement (flat, centered) outvotes
  the rare-but-correct well/clear.
- That is why the GAN adds an explicit line-clear head, and why the transformer
  **scores every legal candidate** instead of generating one.

Next: `02_gan_training_and_solver.ipynb`.
